# Jev emoji-provider trial — completed results

The actual Jev 1.14 trial completed **324 requests for $0.035226030**. The 50-image training pilot checked integration and cost; both Jev roles then ran on all 112 original model disagreements in the reused 1,976-image development set. The complete router scored 1,890/1,976 versus 1,919/1,976 for DINOv2 + ConvNeXt. Conservative bounds exclude both frozen global candidates, so the existing two-model CSV remains recommended.

This notebook defaults to **cached replay with new API calls disabled**. The package contains the real requests, responses and accounting. No key is needed to reproduce the completed result. Jev was used through TypeSafe's hosted API, without fine-tuning its weights. These are exploratory reused-development results, not a new Kaggle score.


## 1. Upload the ready-trial ZIP

Open this notebook in Google Colab and upload the accompanying ready-trial package. For a resumed session, upload the latest resume ZIP instead. Run this upload cell once in a fresh runtime.

In [ ]:
from google.colab import files
from pathlib import Path
import json
import os
import shutil
import stat
import subprocess
import sys
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one ready-trial or resume ZIP.")
archive_name = next(iter(uploaded))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("The uploaded package must be a ZIP file.")
archive_path = Path(archive_name).resolve()
del uploaded

EXTRACT_DIR = Path("/content/jev_trial_workspace")
EXTRACT_DIR.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        destination = (EXTRACT_DIR / member.filename).resolve()
        if not destination.is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError("The ZIP contains an invalid extraction path.")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("The ZIP must contain ordinary files, not symbolic links.")
    archive.extractall(EXTRACT_DIR)

pipeline_paths = list(EXTRACT_DIR.rglob("jev_pipeline.py"))
protocol_paths = list(EXTRACT_DIR.rglob("protocol.json"))
if len(pipeline_paths) != 1 or len(protocol_paths) != 1:
    raise ValueError("Expected one pipeline and one prepared protocol; use the ready-trial package.")
CODE_DIR = pipeline_paths[0].parent
RUN_DIR = protocol_paths[0].parent
if not (CODE_DIR / "jev_client.py").is_file():
    raise ValueError("The package is missing its adjacent Jev client.")
print("Prepared package loaded.")
print("Code:", CODE_DIR)
print("Experiment:", RUN_DIR)


## 2. Install the two pinned dependencies and run offline checks

These commands do not call TypeSafe. The dry run validates the frozen inputs and request shapes; its placeholder probability vector is not a Jev prediction.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "numpy==2.3.5", "Pillow==12.3.0"],
    check=True,
)

def pipeline(*arguments):
    return subprocess.run(
        [sys.executable, str(CODE_DIR / "jev_pipeline.py"), *map(str, arguments)],
        cwd=CODE_DIR,
        check=True,
    )

def save_resume_zip():
    # Only experiment files are archived. Environment variables are not written.
    result = shutil.make_archive("/content/Jev_Trial_Resume", "zip", EXTRACT_DIR)
    files.download(result)
    print("Keep this resume ZIP before ending the runtime.")

if not list(CODE_DIR.glob("test_*.py")):
    raise ValueError("Offline test files are missing from the package.")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(CODE_DIR), "-p", "test_*.py", "-v"],
    cwd=CODE_DIR,
    check=True,
)
pipeline("dry-run", "--run-dir", RUN_DIR)
pipeline("budget", "--run-dir", RUN_DIR)


## 3. Optional credential for new API calls

The delivered completed checkpoint needs no key. Leave `ENABLE_NEW_API_CALLS = False` to replay it. A separately justified new experiment would require a private runtime key and must retain the same spending ledger. The current frozen candidates cannot recover even with perfect remaining predictions.


In [ ]:
ENABLE_NEW_API_CALLS = False

if ENABLE_NEW_API_CALLS:
    import getpass
    runtime_key = getpass.getpass("TypeSafe API key (hidden): ")
    if not runtime_key.strip() or runtime_key != runtime_key.strip():
        raise ValueError("Enter a non-empty key without surrounding whitespace.")
    os.environ["TYPESAFE_API_KEY"] = runtime_key
    del runtime_key
    print("Runtime credential configured; its value is not displayed or saved.")
else:
    os.environ.pop("TYPESAFE_API_KEY", None)
    print("Cached replay mode. No API key configured.")


## 4. Replay the completed 50-image technical pilot

Both stages already exist in the delivered cache. With the API key absent, any unexpected cache miss stops before a request. The pilot was used for integration, timing and cost, not to tune prompts by accuracy. Keep the resulting resume ZIP if you run a future paid session.


In [ ]:
try:
    pipeline("run", "--run-dir", RUN_DIR, "--partition", "pilot", "--limit", "50")
finally:
    try:
        pipeline("budget", "--run-dir", RUN_DIR)
    finally:
        save_resume_zip()


## 5. Replay the 112 completed development disagreements

Both Jev roles are cached for every case where DINOv2 and ConvNeXt disagree. The router is evaluated over all 1,976 development rows because its other 1,864 predictions are unchanged by definition. The standalone and direct-final whole-set scores remain unobserved; the feasibility certificate computes optimistic upper bounds instead.


In [ ]:
try:
    pipeline("run", "--run-dir", RUN_DIR, "--partition", "development", "--limit", "112")
    pipeline("evaluate", "--run-dir", RUN_DIR)
    subprocess.run(
        [sys.executable, str(CODE_DIR / "jev_feasibility.py"), "--run-dir", str(RUN_DIR)],
        cwd=CODE_DIR, check=True,
    )
finally:
    try:
        pipeline("budget", "--run-dir", RUN_DIR)
    finally:
        save_resume_zip()


## 6. Inspect and replay the saved evidence

The completed router failed the promotion gate. The standalone global candidates lack whole-set predictions, but their optimistic accuracy bounds are already below the baseline. No Jev deployment candidate is selected. These commands read the saved results without another API call.


In [ ]:
pipeline("evaluate", "--run-dir", RUN_DIR)
pipeline("budget", "--run-dir", RUN_DIR)
pipeline("replay", "--run-dir", RUN_DIR)


## 7. Optional deployment interface

For this completed experiment no candidate passes, and deployment stays disabled. The existing CSV is in `recommended_submission/`. The code below is retained as the original deployment interface for a separately justified future protocol; it checks the gate and the shared budget and never uploads to Kaggle.


In [ ]:
ENABLE_DEPLOYMENT = False  # Set True only after a candidate passes the evaluation gate.
DATASET_ROOT = "/content/emoji_dataset"  # Use the exact dataset folder described in README.

if ENABLE_DEPLOYMENT:
    try:
        pipeline("deploy", "--run-dir", RUN_DIR, "--dataset-root", DATASET_ROOT)
    finally:
        try:
            pipeline("budget", "--run-dir", RUN_DIR)
        finally:
            save_resume_zip()
else:
    print("Optional deployment is disabled. Development results remain available.")


## 8. Keep the final resume ZIP and clear the runtime key

The ZIP preserves the prompts, ledger, raw responses, and available results. Students should write their course report themselves and disclose the supplied AI-generated code and prompts.

In [ ]:
save_resume_zip()
if "TYPESAFE_API_KEY" in os.environ:
    del os.environ["TYPESAFE_API_KEY"]
print("Runtime API key cleared.")
